In [0]:
spark.sql("USE CATALOG retail_lakehouse")
spark.sql("USE SCHEMA sliver")

In [0]:
# read the data from bronze table
shipping_df = (
    spark.read.table("retail_lakehouse.bronze.shipping")
)

display(shipping_df)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
# standardize columns

shipping_clean_df  = (
    shipping_df
    .withColumn(
        "order_id",
        F.trim(F.col("order_id"))
    )
    .withColumn(
        "shipment_id",
        F.trim(F.col("shipment_id"))
    )
    .withColumn(
        "tracking_number",
        F.trim(F.col("tracking_number"))
    )
    .withColumn(
        "carrier",
        F.upper(F.trim(F.col("carrier")))
    )
    .withColumn(
        "shipment_status",
        F.upper(F.trim(F.col("shipment_status")))
    )
    .withColumn(
        "ship_timestamp",
        F.to_timestamp(F.col("ship_timestamp"), "yyyy-MM-dd HH:mm:ss")
    )
    .withColumn(
        "delivery_timestamp",
        F.to_timestamp(F.col("delivery_timestamp"), "yyyy-MM-dd HH:mm:ss")
    )
)

display(shipping_clean_df)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

window = Window.partitionBy("shipment_id")

# basic validation
shipping_validation_df = (
    shipping_clean_df

    .withColumn(
        "dq_order_id_not_null",
        F.col("order_id").isNotNull()
    )

    .withColumn(
        "dq_shipment_id_not_null",
        F.col("shipment_id").isNotNull()
    )

    .withColumn(
        "dq_carrier_not_null",
        F.col("carrier").isNotNull()
    )

    .withColumn(
        "dq_shipment_status",
        F.col("shipment_status").isNotNull()
    )

    .withColumn(
        "dq_tracking_number",
        F.col("tracking_number").isNotNull()
    )

    .withColumn(
        "dq_ship_timestamp",
        F.col("ship_timestamp").isNotNull()
    )

    .withColumn(
        "dq_delivery_timestamp",
        F.col("delivery_timestamp").isNotNull()
    )

    .withColumn(
        "dq_shipment_id_count",
        F.count("*").over(window)
    )

    .withColumn(
        "dq_shipment_id_unique",
        F.col("dq_shipment_id_count") == 1
    )
)

display(shipping_validation_df)

In [0]:
shipping_lookup_df = (
    spark.read.table("retail_lakehouse.sliver.orders")
    .select(
        F.col("order_id").alias("ref_order_id"),
        F.col("order_timestamp")
    )
    .dropDuplicates(["ref_order_id"])
)

In [0]:
shipping_order_check_df = (
    shipping_validation_df
    .join(
        shipping_lookup_df,
        shipping_validation_df["order_id"] ==
        shipping_lookup_df["ref_order_id"],
        "left"
    )
    .withColumn(
        "dq_order_id_exists",
        F.col("ref_order_id").isNotNull()
    )
)

In [0]:
shipping_time_df = (
    shipping_order_check_df
    .withColumn(
        "dq_ship_after_order",
        F.col("ship_timestamp") >= F.col("order_timestamp")
    )
    .withColumn(
        "dq_delivery_after_ship",
        F.col("delivery_timestamp") >= F.col("ship_timestamp")
    )
)

In [0]:
shipping_time_df = (
    shipping_time_df
    .withColumn(
        "dq_delivery_after_ship",
        F.col("delivery_timestamp") >= F.col("ship_timestamp")
    )
)

In [0]:
shipping_final_df = (
    shipping_time_df
    .withColumn(
        "dq_status",
        F.when(
            F.col("dq_order_id_not_null") &
            F.col("dq_shipment_id_not_null") &
            F.col("dq_carrier_not_null") &
            F.col("dq_shipment_status") &
            F.col("dq_tracking_number") &
            F.col("dq_ship_timestamp") &
            F.col("dq_delivery_timestamp") &
            F.col("dq_shipment_id_unique") &
            F.col("dq_order_id_exists") &
            F.col("dq_ship_after_order") &
            F.col("dq_delivery_after_ship"),
            "PASS"
        ).otherwise("FAIL")
    )
)

In [0]:
valid_df = (
    shipping_final_df
    .filter(F.col("dq_status") == "PASS")
)

invalid_df = (
    shipping_final_df
    .filter(F.col("dq_status") == "FAIL")
)

In [0]:
print("Total:", shipping_final_df.count())
print("Valid:", valid_df.count())
print("Invalid:", invalid_df.count())

In [0]:
silver_shipping_df = (
    valid_df
    .select(
        "shipment_id",
        "order_id",
        "carrier",
        "tracking_number",
        "shipment_status",
        "ship_timestamp",
        "delivery_timestamp"
    )
)

In [0]:
(
    silver_shipping_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("retail_lakehouse.sliver.shipping")
)

In [0]:
(
    invalid_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("retail_lakehouse.sliver.shipping_quarantine")
)

In [0]:
print("shipping_clean_df:", shipping_clean_df.count())
print("shipping_validation_df:", shipping_validation_df.count())
print("shipping_order_check_df:", shipping_order_check_df.count())
print("shipping_time_df:", shipping_time_df.count())
print("shipping_final_df:", shipping_final_df.count())
print("valid_df:", valid_df.count())
print("invalid_df:", invalid_df.count())